<hr>

```text
                                 _         __      _       _                 _    
  /\  /\__ _ _ ____   _____  ___| |_    /\ \ \___ | |_ ___| |__   ___   ___ | | __
 / /_/ / _` | '__\ \ / / _ \/ __| __|  /  \/ / _ \| __/ _ \ '_ \ / _ \ / _ \| |/ /
/ __  / (_| | |   \ V /  __/\__ \ |_  / /\  / (_) | ||  __/ |_) | (_) | (_) |   < 
\/ /_/ \__,_|_|    \_/ \___||___/\__| \_\ \/ \___/ \__\___|_.__/ \___/ \___/|_|\_\
```

<br>
<hr>

### Overview

*Harvest Notebook:* A modular and interactive web scraper designed for flexibility and scalability. This tool allows users to input a list of URLs, check robots.txt files for permissions, scrape data from websites, and generate a suite of comprehensive reports. Reports include page specifications, extracted data, and diagnostics of the scraping process. Built with a Jupyter Notebook interface for user interactivity and Python modules for robust core functionality.

* Developed by: **David Blessent**
* Publisher: **Almond House Publishing**
* Repository: **https://github.com/almondhouse27/harvest-notebook**

If you have any questions, comments, concerns, or just want to talk data, feel free to reach out to me at: `almondhousepublishing@protonmail.com` <br>
I very likely **will be slow** to respond, but *I will get back to you*.<br>
Submit any issues with and suggestions for the Notebook, its code, and its features through the GitHub repository. I welcome feedback and am happy to assist with any inquiries!<br>

<hr>

### How To Use Harvest Notebook

<hr>

### Notebook Setup

In [1]:
"""
imports Harvest Notebook modules from: 
    ./src/

moonlights as a function directory
"""

from src.config      import install_requirements, setup_logging
from src.utils       import complete, get_versions, create_output_directory
from src.input_data  import read_csv
from src.cleaner     import handle_missing_data, input_review
from src.scraper     import harvest
complete("Library & module imports")


Library & module imports complete!


In [2]:
"""
constants define notebook controls
    - defines file paths for input & output with os library
    - provides:
        - csv column headers constraint
        - regular expression for url validation
        - arguments for version-data-collection command loop
"""

import os
import re

INPUT = os.path.join('data', 'input', 'data.csv')
BAD_DATA = os.path.join('data', 'input', 'bad_data.csv')
OUTPUT = os.path.join('data', 'output')
LOG = os.path.join('data', 'log', 'app.log')
URL_CHECK = re.compile(r'^(https?://)?([a-z0-9]+([-\w]*[a-z0-9])*\.)+[a-z]{2,6}(:[0-9]{1,5})?(/.*)?$', re.IGNORECASE)
COLUMNS = [
    'Category',
    'Name', 
    'Url'
]
STACK = [
    ['python', '--version'],
    ['jupyter', '--version']
]
complete("Defining constants")

Defining constants complete!


In [3]:
"""
runs function(s) contained in: 
    ./src/config.py & ./src/utils.py

setup_logging() configures logging, log output accumulates in: 
    ./data/log/app.log

install_requirements() calls pip install requrirements.txt with subprocess library 
    library installs: [pandas, requests] 

get_versions() collects python & jupyter version data for diagnostic report with subprocess library
    returns: version_data
"""

setup_logging(LOG)
install_requirements()
version_data = get_versions(STACK)
complete("Configure notebook")

Configure notebook complete!


<hr>

### Prepare The Dataframe

In [4]:
"""
runs function(s) contained in:
    ./src/input_data.py

read_csv() reads list of URLs from:
    ./data/input/data.csv 
    & checks that formatting of csv data matches column requirements: ['Category', 'Name', 'Url']
        - if not matches: 
            -- copies ./data/input/data.csv to ./data/input/bad_data.csv 
            -- clears ./data/input/data.csv and writes correct column headers
        - if matches:
            -- returns: df (dataframe)
"""

df = read_csv(INPUT, COLUMNS, BAD_DATA)
complete("Reading input data")

Reading input data complete!


In [5]:
"""
runs function contained in:
    ./src/cleaner.py

input review provides an overview of the dataframe's structure and content including:
    - dataframe shape
    - number of duplicated rows
    - count of URLs with http/https
    - unique values in the Category column
    - dataframe info
"""

input_review(df)
complete("Data exploration")

Dataframe shape: (109, 3)
Duplicated rows: 0
Count of http:   0
Count of https:  101
Category values: ['BoardGame' nan 'Software' 'Computer' 'Electronics' 'Guitars'
 'InvestmentFirms' 'Publishing' 'SportsGear' 'Synthesizer' 'Travel'
 'VideoGame']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  107 non-null    object
 1   Name      106 non-null    object
 2   Url       105 non-null    object
dtypes: object(3)
memory usage: 2.7+ KB
None

      Category                    Name                                   Url
0    BoardGame            Leader Games          https://www.leadergames.com/
1    BoardGame    Indie Boards & Cards  https://www.indieboardsandcards.com/
2    BoardGame        Stonemaier Games          https://stonemaiergames.com/
3    BoardGame                  The OP              https://www.theop.games/
4    BoardGame    Fantasy F

In [6]:
"""
runs function contained in:
    ./src/cleaner.py


"""
df, removed_input_rows = handle_missing_data(df, URL_CHECK)
complete("Fill missing values")

Missing Data Summary
---------------------
Total Missing Values: 9

Columns with Missing Values:
Category, Name, Url

Missing Values Per Column:
Category       : 2
Name           : 3
Url            : 4

Missing Url Rows Removed
---------------------
        Category                   Name                       Url
6      BoardGame      https://cmon.com/              CMON Limited
15      Software                    PHP  I love PHP and Composer!
29      Computer                 NVIDIA                       NaN
79   Synthesizer    Teenage Engineering                       NaN
80   Synthesizer              Behringer                       NaN
83   Synthesizer  https://www.korg.com/                       NaN
95        Travel                 Hilton   gttps://www.hilton.com/
101    VideoGame                BioWare   https//www.bioware.com/
---------------------
*Replaced all missing Category and Name values with 'Unavailable'


Missing Data Summary
---------------------
Total Missing Values: 0

<hr>

### Run The Scraper

In [7]:
reports_directory = create_output_directory(OUTPUT)
print(f"Output path: {reports_directory}")
complete("Create timestamped directory for reports")

Output path: data\output\250105-0011
Create timestamped directory for reports complete!


In [8]:
website_words = harvest(df, reports_directory)
website_words
complete("Harvest Notebook web scraper")


Saved chunk 1 with 5962 rows to data\output\250105-0011\raw\website_words_chunk_1.csv
Saved chunk 2 with 7449 rows to data\output\250105-0011\raw\website_words_chunk_2.csv
Saved chunk 3 with 8540 rows to data\output\250105-0011\raw\website_words_chunk_3.csv
Saved chunk 4 with 7870 rows to data\output\250105-0011\raw\website_words_chunk_4.csv
Saved chunk 5 with 0 rows to data\output\250105-0011\raw\website_words_chunk_5.csv
Harvest Notebook web scraper complete!
